# One-time FHIR assessment reset

**Destructive operation — do not add this notebook to `pl_fhir_assessment`.** It is intended for a one-time manual reset only. The recurring pipeline must preserve incremental watermarks, Raw history, and SCD Type 2 history.

By default this notebook is a dry run and deletes nothing. To reset this assessment in the attached `lh_fhir_assignment` Lakehouse, set `CONFIRM_RESET = "RESET_FHIR_ASSESSMENT"`. To also remove the Raw response files under `Files/raw/fhir`, set `DELETE_RAW_FILES = True`. Do not use this on a Lakehouse containing other projects.

In [ ]:
# Safety parameters — defaults are non-destructive.
CONFIRM_RESET = ""
DELETE_RAW_FILES = False

In [ ]:
from pathlib import Path
import shutil

RAW_ROOT = Path("/lakehouse/default/Files/raw/fhir")
LAYER_SCHEMAS = ["bronze", "silver", "gold"]
LEGACY_TABLES = [
    "bronze_fhir_patient", "bronze_fhir_encounter",
    "bronze_fhir_observation", "bronze_fhir_condition",
    "silver_fhir_patient", "silver_fhir_encounter",
    "silver_fhir_observation", "silver_fhir_condition",
    "gold_fhir_patient", "gold_fhir_encounter",
    "gold_fhir_observation", "gold_fhir_condition",
]
CONTROL_TABLES = [
    "dbo.fhir_ingestion_audit",
    "dbo.fhir_ingestion_watermarks",
]
EXPECTED_SCHEMA_TABLES = [
    f"{schema}.{resource}"
    for schema in LAYER_SCHEMAS
    for resource in ["Patient", "Encounter", "Observation", "Condition"]
]

print("=== FHIR assessment reset plan ===")
print("Layer tables:", EXPECTED_SCHEMA_TABLES)
print("Legacy tables:", LEGACY_TABLES)
print("Control tables:", CONTROL_TABLES)
print("Raw folder exists:", RAW_ROOT.exists(), str(RAW_ROOT))
print("Delete Raw files:", bool(DELETE_RAW_FILES))

if CONFIRM_RESET != "RESET_FHIR_ASSESSMENT":
    print("\nDRY RUN ONLY — no data, tables, or schemas were changed.")
    print('To reset the tables, set CONFIRM_RESET = "RESET_FHIR_ASSESSMENT" and rerun.')
    print("To also delete Raw response files, set DELETE_RAW_FILES = True.")
else:
    dropped_tables = []

    # Drop all tables inside the three assessment layer schemas, including any
    # additional exploratory tables in those schemas. This is intentionally destructive.
    existing_schemas = {str(row[0]) for row in spark.sql("SHOW SCHEMAS").collect()}
    for schema_name in LAYER_SCHEMAS:
        if schema_name in existing_schemas:
            schema_tables = spark.sql(f"SHOW TABLES IN {schema_name}").collect()
            for row in schema_tables:
                row_data = row.asDict()
                table_name = row_data.get("tableName") or row_data.get("tablename")
                if table_name:
                    spark.sql(f"DROP TABLE IF EXISTS `{schema_name}`.`{table_name}`")
                    dropped_tables.append(f"{schema_name}.{table_name}")

    # Drop previously used flat/prefixed tables and the control tables.
    for table_name in LEGACY_TABLES + CONTROL_TABLES:
        if spark.catalog.tableExists(table_name):
            spark.sql(f"DROP TABLE IF EXISTS {table_name}")
            dropped_tables.append(table_name)

    # Recreate the medallion schemas empty for a clean first pipeline run.
    for schema_name in LAYER_SCHEMAS:
        try:
            spark.sql(f"DROP SCHEMA IF EXISTS {schema_name}")
        except Exception as exc:
            # If the runtime does not permit dropping an empty schema, keep it and
            # ensure it exists. All tables from that schema were already dropped.
            print(f"Schema {schema_name} kept after DROP SCHEMA attempt: {str(exc)[:250]}")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

    if DELETE_RAW_FILES:
        if RAW_ROOT.exists():
            shutil.rmtree(RAW_ROOT)
            print("Deleted Raw folder:", str(RAW_ROOT))
        else:
            print("Raw folder did not exist; nothing to delete.")
    else:
        print("Raw response files retained by request.")

    print("\nReset completed.")
    print("Tables dropped:", len(dropped_tables))
    for table_name in dropped_tables:
        print(" -", table_name)
    print("Layer schemas recreated/verified:", LAYER_SCHEMAS)
    print("Next: set CONFIRM_RESET back to an empty string before saving.")
